# Generate Conversations with GRPO-Trained Therapist Model

This notebook loads GRPO-trained checkpoints and generates conversations with simulated patients,
in the same format as the DPO experiments, so results can be combined in `Conv_EDA_Unified.ipynb`.

## Notebook Flow

1. **Configuration** — Experiment name, epochs to run, conversation hyperparameters, output paths
2. **Setup** — Installs, imports, authentication
3. **Tokenizer** — Load tokenizer with GRPO-specific ChatML chat template
4. **Model Helpers** — Checkpoint detection and model loading utilities
5. **Conversation Simulation** — Functions to simulate therapist-patient conversations
6. **Patient Permutations** — Generate all 96 patient scenario permutations
7. **Conversation Generation** — Main loop: load model once, swap adapters per epoch, generate and save conversations

---
## 1. Configuration

All tunable parameters in one place.

In [1]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                              CONFIGURATION                                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────────
# Model IDs
# ─────────────────────────────────────────────────────────────────────────────────
BASE_MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"   # Must match adapter's base_model_name_or_path
PATIENT_MODEL_ID = "gpt-4o-mini-2024-07-18"              # OpenAI model for simulated patient
TOKENIZER_ID = "meta-llama/Llama-3.2-1B"


# ─────────────────────────────────────────────────────────────────────────────────
# GRPO Experiment
# ─────────────────────────────────────────────────────────────────────────────────
GRPO_RUN_DIR = "../grpo_runs/V1/GRPO_Oracle-Q1Q2_Llama32-1B-Instruct_LA5_G4"
EPOCHS_TO_RUN       = [3, 6, 9, 12, 15, 18, 21, 24]   # 0 = base model (no adapter); 1+ = checkpoint epochs

# ─────────────────────────────────────────────────────────────────────────────────
# Output
# ─────────────────────────────────────────────────────────────────────────────────
OUTPUT_BASE_DIR = "LLM_DATA/Conversation_with_Eval_V3/GRPO-Instruct"

# ─────────────────────────────────────────────────────────────────────────────────
# Conversation Parameters
# ─────────────────────────────────────────────────────────────────────────────────
MAX_TOKENS_PER_RESPONSE = 200
NUM_UTTERANCES = 49       # Not including therapist's initial utterance (50 total)
TEMPERATURE_THERAPIST = 0.9
TEMPERATURE_PATIENT = 0.7

# ─────────────────────────────────────────────────────────────────────────────────
# Runtime / Retry Controls
# ─────────────────────────────────────────────────────────────────────────────────
BATCH_SIZE = 8
MAX_EPOCH_RETRIES_WITHOUT_PROGRESS = 3
BATCH_COOLDOWN_SECONDS = 1.0

# ─────────────────────────────────────────────────────────────────────────────────
# VRAM safety
# ─────────────────────────────────────────────────────────────────────────────────
THERAPIST_MAX_INPUT_TOKENS = 2048
OOM_FALLBACK_MAX_NEW_TOKENS = 128
OOM_FALLBACK_TEMPERATURE = 0.9

# ─────────────────────────────────────────────────────────────────────────────────
# Quantization
# ─────────────────────────────────────────────────────────────────────────────────
QUANT_LOAD_IN_4BIT = True
QUANT_4BIT_COMPUTE_DTYPE = "bfloat16"   # options: bfloat16, float16, float32
QUANT_4BIT_TYPE = "nf4"

# ─────────────────────────────────────────────────────────────────────────────────
# API resilience
# ─────────────────────────────────────────────────────────────────────────────────
PATIENT_API_MAX_RETRIES = 3
PATIENT_API_BACKOFF_SECONDS = 1
PATIENT_API_CONCURRENCY = 16



# ═══════════════════════════════════════════════════════════════════════════════
# Configuration Summary
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("CONFIGURATION SUMMARY")
print("=" * 70)
print(f"  Base model:         {BASE_MODEL_ID}")
print(f"  Patient model:      {PATIENT_MODEL_ID}")
print(f"  GRPO run dir:       {GRPO_RUN_DIR}")
print(f"  Epochs to run:      {EPOCHS_TO_RUN}")
print("-" * 70)
print(f"  Max new tokens:     {MAX_TOKENS_PER_RESPONSE}")
print(f"  Num utterances:     {NUM_UTTERANCES} (+1 init = {NUM_UTTERANCES + 1} total)")
print(f"  Temp therapist:     {TEMPERATURE_THERAPIST}")
print(f"  Temp patient:       {TEMPERATURE_PATIENT}")
print("-" * 70)
print(f"  Batch size:                     {BATCH_SIZE}")
print(f"  Max retries without progress:   {MAX_EPOCH_RETRIES_WITHOUT_PROGRESS}")
print(f"  Batch cooldown:                 {BATCH_COOLDOWN_SECONDS} seconds")
print("-" * 70)
print(f"  Therapist max input tokens: {THERAPIST_MAX_INPUT_TOKENS}")
print(f"  OOM fallback max tokens:    {OOM_FALLBACK_MAX_NEW_TOKENS}")
print(f"  OOM fallback temperature:   {OOM_FALLBACK_TEMPERATURE}")
print("-" * 70)
print(f"  Quantization: load_in_4bit={QUANT_LOAD_IN_4BIT}, dtype={QUANT_4BIT_COMPUTE_DTYPE}, type={QUANT_4BIT_TYPE}")
print("-" * 70)
print(f"  Patient API retries:        {PATIENT_API_MAX_RETRIES}")
print(f"  Patient API backoff:        {PATIENT_API_BACKOFF_SECONDS} seconds")
print(f"  Patient API concurrency:    {PATIENT_API_CONCURRENCY}")
print("-" * 70)
print(f"  Output base dir:    {OUTPUT_BASE_DIR}")
print("=" * 70)

CONFIGURATION SUMMARY
  Base model:         meta-llama/Llama-3.2-1B-Instruct
  Patient model:      gpt-4o-mini-2024-07-18
  GRPO run dir:       ../grpo_runs/GRPO_Oracle-Q1Q2_Llama32-1B-Instruct_LA5_G4_V2
  Epochs to run:      [3, 6, 9, 12, 15, 18, 21, 24]
----------------------------------------------------------------------
  Max new tokens:     200
  Num utterances:     49 (+1 init = 50 total)
  Temp therapist:     0.9
  Temp patient:       0.7
----------------------------------------------------------------------
  Batch size:                     8
  Max retries without progress:   3
  Batch cooldown:                 1.0 seconds
----------------------------------------------------------------------
  Therapist max input tokens: 2048
  OOM fallback max tokens:    128
  OOM fallback temperature:   0.9
----------------------------------------------------------------------
  Quantization: load_in_4bit=True, dtype=bfloat16, type=nf4
-------------------------------------------------------

---
## 2. Setup

Installs, imports, and authentication.

In [2]:
# %pip install -q transformers trl peft bitsandbytes openai numpy pandas torch fsspec==2025.3.2

In [4]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                                 IMPORTS                                    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Standard library
import os
import sys
import json
import re
import textwrap
import asyncio
import time
from dataclasses import dataclass
from typing import List, Optional, Tuple
import gc

# Async compatibility for Jupyter
import nest_asyncio
nest_asyncio.apply()

# Third-party
import numpy as np
import pandas as pd
import torch

# OpenAI API client (async for batched patient calls)
from openai import AsyncOpenAI

# Resolve shared modules from workspace root
# Walk up from CWD to find the workspace root (depth-independent).
_HELPER_FILES = ['openai_key.txt', 'HF_key.txt']
_cur = os.path.abspath(os.getcwd())
_WORKSPACE_ROOT = None
for _ in range(8):
    if all(os.path.exists(os.path.join(_cur, _hf)) for _hf in _HELPER_FILES):
        _WORKSPACE_ROOT = _cur
        break
    _parent = os.path.dirname(_cur)
    if _parent == _cur:
        break
    _cur = _parent
if _WORKSPACE_ROOT is None:
    raise RuntimeError('Could not locate workspace root with key files')
if _WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, _WORKSPACE_ROOT)

# Custom modules
from system_prompts_builder_V3 import generate_all_permutations, CounselorPersonality

# Transformers and related libraries
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

# Progress bar
from tqdm import tqdm

# Version info
import transformers, trl, peft, bitsandbytes, openai, fsspec

print(f"torch version: {torch.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"trl version: {trl.__version__}")
print(f"peft version: {peft.__version__}")
print(f"bitsandbytes version: {bitsandbytes.__version__}")
print(f"openai version: {openai.__version__}")
print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"fsspec version: {fsspec.__version__}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"CUDA version: {torch.version.cuda}")

# Collect garbage first, then clear CUDA cache
gc.collect()
torch.cuda.empty_cache()

torch version: 2.9.1+cu130
transformers version: 4.57.3
trl version: 0.27.2
peft version: 0.18.1
bitsandbytes version: 0.49.1
openai version: 2.17.0
numpy version: 2.4.0
pandas version: 2.3.3
fsspec version: 2025.10.0
Using device: cuda
CUDA version: 13.0


In [ ]:
import os
import sys
# Walk up from CWD to find the workspace root (depth-independent).
_HELPER_FILES = ['openai_key.txt', 'HF_key.txt']
_cur = os.path.abspath(os.getcwd())
_WORKSPACE_ROOT = None
for _ in range(8):
    if all(os.path.exists(os.path.join(_cur, _hf)) for _hf in _HELPER_FILES):
        _WORKSPACE_ROOT = _cur
        break
    _parent = os.path.dirname(_cur)
    if _parent == _cur:
        break
    _cur = _parent
if _WORKSPACE_ROOT is None:
    raise RuntimeError('Could not locate workspace root with key files')
if _WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, _WORKSPACE_ROOT)
# ─────────────────────────────────────────────────────────────────────────────────
# Authentication
# ─────────────────────────────────────────────────────────────────────────────────

# OpenAI (async client for batched patient calls)
OpenAI_API_KEY = open(os.path.join(_WORKSPACE_ROOT, "openai_key.txt"), "r").read().strip()
client = AsyncOpenAI(api_key=OpenAI_API_KEY)
_patient_sem = asyncio.Semaphore(PATIENT_API_CONCURRENCY)

# HuggingFace
from huggingface_hub import login
hf_token = open(os.path.join(_WORKSPACE_ROOT, "HF_key.txt"), "r").read().strip()
login(token=hf_token)

# # For Colab use:
# from huggingface_hub import login
# from google.colab import userdata
# hf_token = userdata.get("huggingface")
# login(token=hf_token)

---
## 3. Tokenizer

Load tokenizer from the GRPO run directory, which includes the correct ChatML-style chat template
(with spaces around `<|im_start|>` / `<|im_end|>`) used during GRPO training.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                               TOKENIZER                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.truncation_side = "left"

# Custom ChatML-style template
tokenizer.chat_template = (
    "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}"
    "{% for message in messages %}"
    "{{'<|im_start|> ' + message['role'] + '\n' + message['content'] + ' <|im_end|>' + '\n'}}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|im_start|> assistant\n' }}{% endif %}"
)

print(f"✓ Tokenizer loaded: {TOKENIZER_ID}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})")
print(f"  Special tokens: {tokenizer.special_tokens_map}")

# ─────────────────────────────────────────────────────────────────────────────────
# Verify chat template
# ─────────────────────────────────────────────────────────────────────────────────
print("=" * 70)
test_msgs = [
    {"role": "system", "content": "you are a robot"},
    {"role": "user", "content": "Hello, how is it to be a robot?"},
    {"role": "assistant", "content": "It is great to be a robot."},
    {"role": "user", "content": "cool."},
]
chat = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
print(chat)

---
## 4. Model Helpers

Checkpoint detection and model loading utilities.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                         MODEL HELPER FUNCTIONS                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────────
# Checkpoint Detection
# ─────────────────────────────────────────────────────────────────────────────────

def list_local_checkpoints(output_dir: str) -> list:
    """List all checkpoint directories sorted by step number.

    Returns:
        List of (step, path) tuples, sorted ascending by step.
    """
    if not os.path.isdir(output_dir):
        return []
    checkpoints = []
    for name in os.listdir(output_dir):
        if name.startswith("checkpoint-") and os.path.isdir(os.path.join(output_dir, name)):
            try:
                step = int(name.split("-")[-1])
                checkpoints.append((step, os.path.join(output_dir, name)))
            except ValueError:
                continue
    checkpoints.sort(key=lambda x: x[0])
    return checkpoints


def get_checkpoint_by_epoch(output_dir: str, epoch: int):
    """Get checkpoint path by epoch index (1-indexed).

    Returns:
        (step, path) tuple, or None if epoch is out of range.
    """
    checkpoints = list_local_checkpoints(output_dir)
    if 1 <= epoch <= len(checkpoints):
        return checkpoints[epoch - 1]
    return None


# ─────────────────────────────────────────────────────────────────────────────────
# Model Loading
# ─────────────────────────────────────────────────────────────────────────────────

def load_base_model(base_model_id: str, quantization_config):
    """Load the base model once with quantization.

    Returns:
        The base AutoModelForCausalLM (no adapter attached).
    """
    print(f"  Loading base model: {base_model_id}")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        attn_implementation="sdpa",
    )
    print("  Base model loaded successfully")
    return base_model


def build_multi_adapter_model(base_model, grpo_run_dir: str, epochs: list):
    """Build a single PeftModel with multiple adapters loaded for efficient epoch sweeping.

    Wraps base_model ONCE and loads each checkpoint as a named adapter,
    allowing runtime switching via set_adapter() / disable_adapter().

    Args:
        base_model: The base model (no adapter attached).
        grpo_run_dir: Directory containing GRPO checkpoints.
        epochs: List of epoch numbers. 0 = base model (no adapter), 1+ = checkpoint epochs.

    Returns:
        (model, labels_by_epoch) where:
        - model: PeftModel with all adapters loaded, or base_model if only epoch 0 requested.
        - labels_by_epoch: Dict mapping epoch -> label string (e.g., {0: "base", 3: "epoch_3"}).
    """
    labels_by_epoch = {}
    non_zero_epochs = [e for e in epochs if e != 0]

    for epoch in epochs:
        labels_by_epoch[epoch] = "base" if epoch == 0 else f"epoch_{epoch}"

    # If only base model requested, return as-is
    if not non_zero_epochs:
        print("  Only base model requested — no adapters to load")
        base_model.eval()
        return base_model, labels_by_epoch

    # Resolve checkpoint paths for all non-zero epochs
    epoch_to_path = {}
    for epoch in non_zero_epochs:
        ckpt_info = get_checkpoint_by_epoch(grpo_run_dir, epoch)
        if ckpt_info is None:
            raise FileNotFoundError(
                f"Checkpoint for epoch {epoch} not found in {grpo_run_dir}!"
            )
        _, ckpt_path = ckpt_info
        epoch_to_path[epoch] = ckpt_path

    # First adapter: wrap base model with PeftModel
    first_epoch = non_zero_epochs[0]
    first_path = epoch_to_path[first_epoch]
    first_label = labels_by_epoch[first_epoch]

    print(f"  Creating PeftModel from epoch {first_epoch} (adapter: '{first_label}')")
    multi_model = PeftModel.from_pretrained(
        base_model,
        first_path,
        adapter_name=first_label,
        is_trainable=False,
    )

    # Subsequent adapters: use load_adapter()
    for epoch in non_zero_epochs[1:]:
        ckpt_path = epoch_to_path[epoch]
        label = labels_by_epoch[epoch]
        print(f"  Loading adapter for epoch {epoch} (adapter: '{label}')")
        multi_model.load_adapter(ckpt_path, adapter_name=label, is_trainable=False)

    multi_model.eval()
    print(f"  All adapters loaded: {list(multi_model.peft_config.keys())}")
    return multi_model, labels_by_epoch


# ═══════════════════════════════════════════════════════════════════════════════
# Print Available Checkpoints
# ═══════════════════════════════════════════════════════════════════════════════
available = list_local_checkpoints(GRPO_RUN_DIR)
assert len(available) > 0, f"No checkpoints found in {GRPO_RUN_DIR}"

print(f"Available checkpoints ({len(available)}):")
for step, ckpt_path in available:
    meta_path = os.path.join(ckpt_path, "experiment_metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        print(f"  {os.path.basename(ckpt_path):>20s}  ->  epoch {meta.get('epoch', '?')},  step {meta.get('global_step', '?')}")
    else:
        print(f"  {os.path.basename(ckpt_path):>20s}  ->  step {step}")

# Validate requested epochs
for ep in EPOCHS_TO_RUN:
    assert ep == 0 or 1 <= ep <= len(available), f"Epoch {ep} out of range (0 or 1-{len(available)})"
print(f"\nRequested epochs: {EPOCHS_TO_RUN}  ✓")

---
## 5. Conversation Simulation

Functions to simulate a multi-turn conversation between a therapist (local model) and
a patient (OpenAI API).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                     CONVERSATION HELPER FUNCTIONS                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────────
# Text Utilities
# ─────────────────────────────────────────────────────────────────────────────────

def reconstruct_conversation_text(utterances: List[str]) -> str:
    return "\n".join([f"[{'THERAPIST' if i % 2 == 0 else 'PATIENT'}]: {utt}" for i, utt in enumerate(utterances)])


def print_conversation(conversation, max_width=80):
    """Print the conversation with roles labeled as [THERAPIST] and [PATIENT]."""
    for i, message in enumerate(conversation):
        role = "[THERAPIST]" if i % 2 == 0 else "[PATIENT]"
        print(f"{role}: \n{textwrap.fill(message, width=max_width)} \n")


# ─────────────────────────────────────────────────────────────────────────────────
# Async Patient Generation (Batched)
# ─────────────────────────────────────────────────────────────────────────────────

async def generate_patient_response_async(
    client,
    model_id,
    messages_Patient_assist,
    max_tokens,
    temperature,
    max_retries=PATIENT_API_MAX_RETRIES,
    backoff_seconds=PATIENT_API_BACKOFF_SECONDS,
 ):
    """Generate a single patient response using the async OpenAI API with retry/backoff."""
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            async with _patient_sem:
                response = await client.chat.completions.create(
                    model=model_id,
                    messages=messages_Patient_assist,
                    max_tokens=max_tokens,
                    temperature=temperature,
                    seed=42
                )
            return response.choices[0].message.content
        except Exception as e:
            last_error = e
            if attempt >= max_retries:
                break
            sleep_s = backoff_seconds * (2 ** (attempt - 1))
            print(f"  Patient API attempt {attempt}/{max_retries} failed: {e}. Retrying in {sleep_s:.1f}s...")
            await asyncio.sleep(sleep_s)

    raise RuntimeError(f"Patient API failed after {max_retries} retries: {last_error}")


async def generate_patient_responses_batch(client, model_id, batch_messages, max_tokens, temperature):
    """Generate patient responses for a batch of conversations concurrently.

    Args:
        client: AsyncOpenAI client instance.
        model_id: OpenAI model ID.
        batch_messages: List of messages_Patient_assist lists (one per conversation).
        max_tokens: Maximum tokens to generate.
        temperature: Sampling temperature.

    Returns:
        List[str]: Patient responses in same order as batch_messages.
    """
    tasks = [
        generate_patient_response_async(client, model_id, messages, max_tokens, temperature)
        for messages in batch_messages
    ]
    return await asyncio.gather(*tasks)


# ─────────────────────────────────────────────────────────────────────────────────
# Batched Therapist Generation
# ─────────────────────────────────────────────────────────────────────────────────

def generate_therapist_responses_batch(
    therapist_model,
    therapist_tokenizer,
    batch_messages,
    max_tokens,
    temperature,
    max_input_tokens=THERAPIST_MAX_INPUT_TOKENS,
 ) -> Tuple[Optional[List[str]], Optional[str]]:
    """Generate one therapist response per conversation using batched GPU inference.

    Args:
        therapist_model: The therapist model to generate responses.
        therapist_tokenizer: The tokenizer (must have padding_side='left').
        batch_messages: List of messages_Therapist_assist lists (one per conversation).
        max_tokens: Maximum number of new tokens to generate per response.
        temperature: Temperature for response generation.
        max_input_tokens: Maximum number of input tokens allowed per prompt.

    Returns:
        (responses, error_type):
            - responses is List[str] on success, else None.
            - error_type is one of: None, "oom", "runtime_error".
    """
    prompts = [
        therapist_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        for messages in batch_messages
    ]

    encoded = None
    outputs = None
    try:
        encoded = therapist_tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_tokens,
            add_special_tokens=False
        ).to(therapist_model.device)

        prompt_lengths = encoded.attention_mask.sum(dim=1).tolist()
        if prompt_lengths:
            print(
                f"  Therapist prompt tokens (active={len(batch_messages)}): "
                f"max={max(prompt_lengths)}, avg={np.mean(prompt_lengths):.1f}"
            )

        with torch.inference_mode():
            outputs = therapist_model.generate(
                input_ids=encoded.input_ids,
                attention_mask=encoded.attention_mask,
                do_sample=True,
                max_new_tokens=max_tokens,
                pad_token_id=therapist_tokenizer.eos_token_id,
                eos_token_id=therapist_tokenizer.eos_token_id,
                temperature=temperature,
                num_return_sequences=1,
                stop_strings=["<|im_end|>"],
                tokenizer=therapist_tokenizer
            )
    except torch.OutOfMemoryError as e:
        print(f"  CUDA OOM during therapist generation: {e}")
        gc.collect()
        torch.cuda.empty_cache()
        return None, "oom"
    except RuntimeError as e:
        msg = str(e).lower()
        if "out of memory" in msg or "cuda" in msg and "memory" in msg:
            print(f"  Runtime CUDA memory failure during therapist generation: {e}")
            gc.collect()
            torch.cuda.empty_cache()
            return None, "oom"
        print(f"  Runtime error during therapist generation: {e}")
        return None, "runtime_error"

    # Use padded input length (not per-item attention_mask sum) to correctly
    # slice out only the newly generated tokens. With left-padding, pad tokens
    # sit at the start, so the real content + generated tokens are at the end.
    # encoded.input_ids.shape[1] gives the full padded prompt length, which is
    # the correct offset to skip past ALL input tokens (pad + real).
    padded_input_length = encoded.input_ids.shape[1]

    responses = []
    for i in range(len(batch_messages)):
        new_tokens = outputs[i][padded_input_length:]

        decoded = therapist_tokenizer.decode(new_tokens, skip_special_tokens=True)
        cleaned = decoded.split("<|im_end|>")[0].strip()
        responses.append(cleaned)

    del encoded, outputs
    gc.collect()
    torch.cuda.empty_cache()

    return responses, None


# ─────────────────────────────────────────────────────────────────────────────────
# Session Management
# ─────────────────────────────────────────────────────────────────────────────────

def handle_session_end(response_content, turn_index):
    """Handle 'SESSION ENDED' keyword in response content."""
    session_ended_keyword = "SESSION ENDED"
    idx = response_content.upper().find(session_ended_keyword)

    if idx != -1:
        print(f"  Detected '{session_ended_keyword}' in response content.")
        session_endded_explanation = response_content[idx + len(session_ended_keyword):]
        response_content = response_content[:idx]
        session_endded_by = "patient" if turn_index % 2 == 0 else "therapist"
        print("  Response content (SESSION ENDED):", response_content)
        print("  Explanation:", session_endded_explanation)
        print("  Session ended by:", session_endded_by)
        return session_endded_by, session_endded_explanation, response_content
    else:
        raise ValueError("SESSION ENDED keyword not found in response content")


def update_conversation(conversation, messages_Patient_assist, messages_Therapist_assist, role_Patient, role_Therapist, response_content):
    """Append a new response to conversation and message lists."""
    conversation.append(response_content)
    messages_Patient_assist.append({"role": role_Patient, "content": response_content})
    messages_Therapist_assist.append({"role": role_Therapist, "content": response_content})


def initialize_conversation(system_prompt_therapist, system_prompt_patient, therapist_init_utterance, patient_init_utterance):
    """Initialize conversation and message lists for both assistants."""
    conversation = [therapist_init_utterance]

    messages_Patient_assist = [
        {"role": "system", "content": system_prompt_patient},
        {"role": "user", "content": therapist_init_utterance}
    ]

    messages_Therapist_assist = [
        {"role": "system", "content": system_prompt_therapist},
        {"role": "user", "content": patient_init_utterance},
        {"role": "assistant", "content": therapist_init_utterance}
    ]

    return conversation, messages_Patient_assist, messages_Therapist_assist

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                  BATCHED CONVERSATION LOOP & SYNTHESIS                     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

@dataclass
class ConversationState:
    """State for a single conversation in a batch."""
    permutation_index: int
    conversation: list
    messages_Patient_assist: list
    messages_Therapist_assist: list
    session_endded_by: Optional[str] = None
    session_endded_explanation: Optional[str] = None
    is_active: bool = True
    failed: bool = False


async def conversation_loop_batch(
    batch_states: List[ConversationState],
    therapist_model,
    therapist_tokenizer,
    client,
    model_id="gpt-4o-mini-2024-07-18",
    max_tokens=100,
    num_utterances=6,
    temperature_therapist=0.7,
    temperature_patient=0.7,
    therapist_max_input_tokens=THERAPIST_MAX_INPUT_TOKENS,
    verbose_responses=False,
    verbose_conversation=False,
):
    """Simulate conversation loops for a batch of conversations.

    Each turn processes only the active (not ended) conversations:
    - Even turns: async concurrent OpenAI calls for all active patients.
    - Odd turns:  batched GPU inference for all active therapists.

    When conversations end mid-batch (SESSION ENDED), their message lists are
    cleared and CUDA cache is reclaimed so subsequent turns use less GPU memory.

    Returns:
        (states, error_type): error_type is None on success, or a failure string.
    """
    for turn_index in range(num_utterances):
        active_states = [s for s in batch_states if s.is_active]
        if not active_states:
            break

        num_active_before = len(active_states)

        if turn_index % 2 == 0:
            role_Patient = "assistant"
            role_Therapist = "user"

            batch_patient_messages = [s.messages_Patient_assist for s in active_states]
            responses = await generate_patient_responses_batch(
                client, model_id, batch_patient_messages, max_tokens, temperature_patient
            )
            if verbose_responses:
                print(f"Patient responses: {responses}")

            for state, response_content in zip(active_states, responses):
                if "SESSION ENDED" in response_content.upper():
                    try:
                        endded_by, endded_expl, response_content = handle_session_end(response_content, turn_index)
                        state.session_endded_by = endded_by
                        state.session_endded_explanation = endded_expl
                        state.is_active = False
                        if response_content:
                            update_conversation(
                                state.conversation, state.messages_Patient_assist,
                                state.messages_Therapist_assist, role_Patient, role_Therapist,
                                response_content
                            )
                    except ValueError:
                        state.is_active = False
                else:
                    update_conversation(
                        state.conversation, state.messages_Patient_assist,
                        state.messages_Therapist_assist, role_Patient, role_Therapist,
                        response_content
                    )
        else:
            role_Patient = "user"
            role_Therapist = "assistant"

            batch_therapist_messages = [s.messages_Therapist_assist for s in active_states]
            responses, error_type = generate_therapist_responses_batch(
                therapist_model, therapist_tokenizer, batch_therapist_messages,
                max_tokens, temperature_therapist,
                max_input_tokens=therapist_max_input_tokens,
            )
            if verbose_responses:
                print(f"Therapist responses: {responses}")
            if responses is None:
                print(
                    f"  Batch therapist generation failed (reason={error_type}) "
                    f"— marking active conversations as failed."
                )
                for state in active_states:
                    state.is_active = False
                    state.failed = True
                return batch_states, (error_type or "therapist_generation_failed")

            for state, response_content in zip(active_states, responses):
                if "SESSION ENDED" in response_content.upper():
                    try:
                        endded_by, endded_expl, response_content = handle_session_end(response_content, turn_index)
                        state.session_endded_by = endded_by
                        state.session_endded_explanation = endded_expl
                        state.is_active = False
                        if response_content:
                            update_conversation(
                                state.conversation, state.messages_Patient_assist,
                                state.messages_Therapist_assist, role_Patient, role_Therapist,
                                response_content
                            )
                        # print conversation after session end
                        if verbose_conversation:
                            print(f"\nFull conversation for permutation index {state.permutation_index} (ended by {state.session_endded_by}):")
                            print_conversation(state.conversation)
                    except ValueError:
                        state.is_active = False
                else:
                    update_conversation(
                        state.conversation, state.messages_Patient_assist,
                        state.messages_Therapist_assist, role_Patient, role_Therapist,
                        response_content
                    )

        # ── Cleanup ended conversations ──────────────────────────────────
        # ── Cleanup ended conversations (CPU memory only) ────────────
        # Free message lists of ended conversations (no longer needed for
        # generation — state.conversation is preserved for saving).
        # No GPU cleanup here — tensors are already freed inside
        # generate_therapist_responses_batch after each turn.
        newly_ended = [s for s in active_states if not s.is_active]
        if newly_ended:
            for s in newly_ended:
                s.messages_Patient_assist = []
                s.messages_Therapist_assist = []
            num_active_after = len([s for s in batch_states if s.is_active])
            print(
                f"  Turn {turn_index}: batch reduced {num_active_before} → "
                f"{num_active_after} active conversations"
            )

    return batch_states, None


async def synthesize_conversations_batch(
    permutations_batch,
    system_prompt_therapist,
    therapist_init_utterance,
    therapist_model,
    therapist_tokenizer,
    client,
    model_id="gpt-4o-mini-2024-07-18",
    max_tokens=50,
    num_utterances=6,
    temperature_therapist=0.7,
    temperature_patient=0.7,
    therapist_max_input_tokens=THERAPIST_MAX_INPUT_TOKENS,
    verbose_responses=False,
    verbose_conversation=False,
):
    """Synthesize a batch of conversations in parallel.

    Returns:
        (states, error_type): error_type is None on success, or a failure string.
    """
    batch_states = []
    for perm_idx, perm in permutations_batch:
        conversation, messages_Patient, messages_Therapist = initialize_conversation(
            system_prompt_therapist,
            perm["patient_system_prompt"],
            therapist_init_utterance,
            ""
        )
        batch_states.append(ConversationState(
            permutation_index=perm_idx,
            conversation=conversation,
            messages_Patient_assist=messages_Patient,
            messages_Therapist_assist=messages_Therapist,
        ))

    print(f"  Batch of {len(batch_states)} conversations — indices {[s.permutation_index for s in batch_states]}")

    return await conversation_loop_batch(
        batch_states,
        therapist_model,
        therapist_tokenizer,
        client,
        model_id,
        max_tokens,
        num_utterances,
        temperature_therapist,
        temperature_patient,
        therapist_max_input_tokens=therapist_max_input_tokens,
        verbose_responses=verbose_responses,
        verbose_conversation=verbose_conversation,
    )

---
## 6. Patient Permutations

Generate all 96 patient scenario permutations and set up the therapist personality (Good level).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                         PATIENT PERMUTATIONS                               ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

permutations = generate_all_permutations(only_expert_therapist=True)
print(f"Number of permutations: {len(permutations)}")
print(f"Permutation keys: {permutations[0].keys()}")
permutation_pd = pd.DataFrame(permutations)

# ─────────────────────────────────────────────────────────────────────────────────
# Therapist Personality Setup (Good level)
# ─────────────────────────────────────────────────────────────────────────────────
good_level = CounselorPersonality.PersonalityLevel.Good
therapist = CounselorPersonality.choose_random_therapist_name()
therapist_good_system_prompt = CounselorPersonality.build_system_prompt(
    personality_level=good_level, name=therapist['name']
)
therapist_good_init_utterance = CounselorPersonality.get_init_utterance(
    personality_level=good_level, name=therapist['name']
)

print(f"Therapist name:           {therapist['name']}")
print(f"Therapist init utterance: {therapist_good_init_utterance}")
print(f"\nExample Patient system prompt:\n{permutations[0]["patient_system_prompt"]}")
print(f"\nExample Patient args:\n{permutations[0]["args"]}")


---
## 7. Conversation Generation

Main loop: for each specified epoch, load the GRPO model checkpoint, generate 96 conversations
(one per patient permutation), and save them as CSV files.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        QUANTIZATION CONFIGURATION                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

_dtype_map = {
    "bfloat16": torch.bfloat16,
    "float16": torch.float16,
    "float32": torch.float32,
}
if QUANT_4BIT_COMPUTE_DTYPE not in _dtype_map:
    raise ValueError(
        f"Unsupported QUANT_4BIT_COMPUTE_DTYPE={QUANT_4BIT_COMPUTE_DTYPE}. "
        f"Use one of {list(_dtype_map.keys())}."
    )

quantization_config = BitsAndBytesConfig(
    load_in_4bit=QUANT_LOAD_IN_4BIT,
    bnb_4bit_compute_dtype=_dtype_map[QUANT_4BIT_COMPUTE_DTYPE],
    bnb_4bit_quant_type=QUANT_4BIT_TYPE
)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        EPOCH EXECUTION HELPERS                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def build_epoch_output_dir(output_base_dir, epoch_num, temp_therapist, temp_patient):
    """Build and create output directory for one epoch configuration."""
    if epoch_num == 0:
        config_folder = f"GRPO_Base_TT{temp_therapist}_TP{temp_patient}"
    else:
        config_folder = f"GRPO_Epoch{epoch_num}_TT{temp_therapist}_TP{temp_patient}"
    path_to_save = os.path.join(output_base_dir, config_folder)
    os.makedirs(path_to_save, exist_ok=True)
    return path_to_save


def get_remaining_indices(path_to_save, total_permutations):
    """Return permutation indices that still need generation."""
    return [
        i for i in range(total_permutations)
        if not os.path.exists(os.path.join(path_to_save, f"conversation_{i}.csv"))
    ]


def run_synthesis_batch(
    permutations_batch,
    therapist_model,
    therapist_tokenizer,
    client,
    patient_model_id,
    max_tokens,
    num_utterances,
    temperature_therapist,
    temperature_patient,
    therapist_system_prompt,
    therapist_init_utterance,
    therapist_max_input_tokens,
    verbose_responses=False,
    verbose_conversation=False,
 ):
    """Execute one batch synthesis call with error handling.

    Returns:
        dict with keys:
            - final_states: list[ConversationState] | None
            - error_type: None | str
    """
    try:
        final_states, error_type = asyncio.get_event_loop().run_until_complete(
            synthesize_conversations_batch(
                permutations_batch=permutations_batch,
                system_prompt_therapist=therapist_system_prompt,
                therapist_init_utterance=therapist_init_utterance,
                therapist_model=therapist_model,
                therapist_tokenizer=therapist_tokenizer,
                client=client,
                model_id=patient_model_id,
                max_tokens=max_tokens,
                num_utterances=num_utterances,
                temperature_therapist=temperature_therapist,
                temperature_patient=temperature_patient,
                therapist_max_input_tokens=therapist_max_input_tokens,
                verbose_responses=verbose_responses,
                verbose_conversation=verbose_conversation,
            )
        )
        return {"final_states": final_states, "error_type": error_type}
    except torch.OutOfMemoryError as e:
        print(f"  ERROR: CUDA OOM in batch generation: {e}")
        gc.collect()
        torch.cuda.empty_cache()
        return {"final_states": None, "error_type": "oom"}
    except RuntimeError as e:
        msg = str(e).lower()
        if "out of memory" in msg or "cuda" in msg and "memory" in msg:
            print(f"  ERROR: Runtime CUDA memory failure in batch generation: {e}")
            gc.collect()
            torch.cuda.empty_cache()
            return {"final_states": None, "error_type": "oom"}
        print(f"  ERROR: Batch generation runtime error: {e}")
        print("  Will retry in next iteration.")
        return {"final_states": None, "error_type": "runtime_error"}
    except Exception as e:
        print(f"  ERROR: Batch generation failed: {e}")
        print("  Will retry in next iteration.")
        return {"final_states": None, "error_type": "batch_exception"}


def save_batch_conversations(final_states, path_to_save):
    """Save successful conversations from a batch. Returns number of files saved."""
    saved_this_batch = 0
    for state in final_states:
        if state.failed or not state.conversation or len(state.conversation) <= 1:
            print(f"  Skipping permutation {state.permutation_index} (failed or empty), will retry.")
            continue

        save_path = os.path.join(path_to_save, f"conversation_{state.permutation_index}.csv")
        pd_conversation = pd.DataFrame({
            "conversation": state.conversation,
            "session_endded_by": state.session_endded_by,
            "session_endded_explanation": state.session_endded_explanation,
        })
        pd_conversation.to_csv(save_path, index=False)
        saved_this_batch += 1
        print(f"  Saved conversation_{state.permutation_index}.csv ({len(state.conversation)} utterances)")

    return saved_this_batch


def run_epoch_generation(
    path_to_save,
    therapist_model,
    therapist_tokenizer,
    client,
    permutations,
    total_permutations,
    batch_size,
    max_epoch_retries_without_progress,
    batch_cooldown_seconds,
    patient_model_id,
    max_tokens_per_response,
    num_utterances,
    temperature_therapist,
    temperature_patient,
    therapist_system_prompt,
    therapist_init_utterance,
    therapist_max_input_tokens,
    oom_fallback_max_new_tokens,
    oom_fallback_temperature,
    verbose_responses=False,
    verbose_conversation=False,
 ):
    """Generate and save all conversations for a single epoch with retries."""
    retries_without_progress = 0
    adaptive_profile_active = False
    oom_failures_total = 0

    while True:
        remaining_indices = get_remaining_indices(path_to_save, total_permutations)
        if not remaining_indices:
            print(f"\n  All {total_permutations} conversations completed!")
            break

        if retries_without_progress >= max_epoch_retries_without_progress:
            print(
                f"\n  WARNING: Reached retry limit without progress "
                f"({max_epoch_retries_without_progress}). Stopping epoch early."
            )
            break

        current_max_tokens = oom_fallback_max_new_tokens if adaptive_profile_active else max_tokens_per_response
        current_temp_therapist = oom_fallback_temperature if adaptive_profile_active else temperature_therapist

        print(f"\n  Remaining: {len(remaining_indices)}/{total_permutations} conversations")
        print(
            f"  Active generation profile: "
            f"max_new_tokens={current_max_tokens}, "
            f"temp_therapist={current_temp_therapist}, "
            f"max_input_tokens={therapist_max_input_tokens}"
        )

        progress_made_this_pass = False
        oom_seen_this_pass = False

        for batch_start in range(0, len(remaining_indices), batch_size):
            batch_indices = remaining_indices[batch_start:batch_start + batch_size]
            print(f"\n{'=' * 70}")
            print(f"  Batch: permutations {batch_indices}")
            print(f"{'=' * 70}")

            permutations_batch = [(idx, permutations[idx]) for idx in batch_indices]
            batch_result = run_synthesis_batch(
                permutations_batch=permutations_batch,
                therapist_model=therapist_model,
                therapist_tokenizer=therapist_tokenizer,
                client=client,
                patient_model_id=patient_model_id,
                max_tokens=current_max_tokens,
                num_utterances=num_utterances,
                temperature_therapist=current_temp_therapist,
                temperature_patient=temperature_patient,
                therapist_system_prompt=therapist_system_prompt,
                therapist_init_utterance=therapist_init_utterance,
                therapist_max_input_tokens=therapist_max_input_tokens,
                verbose_responses=verbose_responses,
                verbose_conversation=verbose_conversation,
            )

            final_states = batch_result["final_states"]
            error_type = batch_result["error_type"]

            if final_states is None:
                if error_type == "oom":
                    oom_seen_this_pass = True
                    oom_failures_total += 1
                    print("  OOM detected in this batch. Skipping and continuing; conversations retry next pass.")
                else:
                    print(f"  Batch failed with reason={error_type}. Skipping and continuing.")

                gc.collect()
                torch.cuda.empty_cache()
                time.sleep(batch_cooldown_seconds)
                continue

            saved_this_batch = save_batch_conversations(final_states, path_to_save)
            if saved_this_batch > 0:
                progress_made_this_pass = True

            time.sleep(batch_cooldown_seconds)

        if progress_made_this_pass:
            retries_without_progress = 0
            if adaptive_profile_active and not oom_seen_this_pass:
                adaptive_profile_active = False
                print("\n  Recovery: no OOM this pass, switching back to primary generation profile.")
        else:
            retries_without_progress += 1
            print(
                f"\n  No new files saved this pass. "
                f"Retry counter: {retries_without_progress}/{max_epoch_retries_without_progress}"
            )
            if oom_seen_this_pass and not adaptive_profile_active:
                adaptive_profile_active = True
                print(
                    f"  Activating OOM fallback profile for next pass: "
                    f"max_new_tokens={oom_fallback_max_new_tokens}, "
                    f"temp_therapist={oom_fallback_temperature}"
                )

    print(f"  OOM events observed this epoch: {oom_failures_total}")


def print_epoch_completion(path_to_save, epoch_num, total_permutations):
    """Print completion summary for one epoch."""
    completed = sum(
        1 for i in range(total_permutations)
        if os.path.exists(os.path.join(path_to_save, f"conversation_{i}.csv"))
    )
    missing = [
        i for i in range(total_permutations)
        if not os.path.exists(os.path.join(path_to_save, f"conversation_{i}.csv"))
    ]

    label = "base" if epoch_num == 0 else f"epoch {epoch_num}"
    print(f"\n  Completed {label}: {completed}/{total_permutations} conversations saved.")
    if missing:
        print(f"  Missing conversation indices ({len(missing)}): {missing}")
    else:
        print(f"  ✓ All conversations generated for {label}.")

### Runtime Safety Notes (OOM Handling)

- Therapist generation now enforces `THERAPIST_MAX_INPUT_TOKENS` on every batched inference call.
- If CUDA OOM occurs in a batch, the batch is skipped (not fatal), cache is cleared, and missing conversations are retried in later passes.
- When a pass has no progress and OOM was seen, the next pass uses fallback settings: `OOM_FALLBACK_MAX_NEW_TOKENS` and `OOM_FALLBACK_TEMPERATURE`.
- After a stable pass with no OOM, the loop automatically returns to the primary generation profile.
- Resume behavior is unchanged: existing `conversation_{i}.csv` files are detected and skipped automatically.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                    MAIN CONVERSATION GENERATION LOOP                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

total_permutations = len(permutations)
has_adapters = any(ep != 0 for ep in EPOCHS_TO_RUN)

# ─────────────────────────────────────────────────────────────────────────────────
# Load base model ONCE
# ─────────────────────────────────────────────────────────────────────────────────
base_model = load_base_model(BASE_MODEL_ID, quantization_config)

# ─────────────────────────────────────────────────────────────────────────────────
# Attach all adapters at once (or keep raw base if only epoch 0)
# ─────────────────────────────────────────────────────────────────────────────────
therapist_model, labels_by_epoch = build_multi_adapter_model(
    base_model, GRPO_RUN_DIR, EPOCHS_TO_RUN
)
therapist_model.config.use_cache = True # Enable caching for faster generation
print(f"  Labels by epoch: {labels_by_epoch}")

# ─────────────────────────────────────────────────────────────────────────────────
# Generate conversations for each epoch
# ─────────────────────────────────────────────────────────────────────────────────

def _run_single_epoch(epoch, therapist_model, labels_by_epoch, verbose_responses=False, verbose_conversation=False):
    """Run conversation generation for a single epoch."""
    epoch_num = epoch
    label = labels_by_epoch[epoch]

    path_to_save = build_epoch_output_dir(
        output_base_dir=OUTPUT_BASE_DIR,
        epoch_num=epoch_num,
        temp_therapist=TEMPERATURE_THERAPIST,
        temp_patient=TEMPERATURE_PATIENT,
    )

    print("=" * 70)
    print(f"{'BASE MODEL' if epoch == 0 else f'EPOCH {epoch_num}'} — adapter: '{label}'")
    print("=" * 70)
    print(f"  Output dir:   {path_to_save}")
    print(f"  Batch size:   {BATCH_SIZE}")
    print(f"  Therapist max input tokens: {THERAPIST_MAX_INPUT_TOKENS}")
    print(
        f"  OOM fallback profile: max_new_tokens={OOM_FALLBACK_MAX_NEW_TOKENS}, "
        f"temp_therapist={OOM_FALLBACK_TEMPERATURE}"
    )
    print("-" * 70)

    # Switch adapter (or use base)
    if epoch != 0 and has_adapters:
        therapist_model.set_adapter(label)
        print(f"  Active adapter: {therapist_model.active_adapters}")

    run_epoch_generation(
        path_to_save=path_to_save,
        therapist_model=therapist_model,
        therapist_tokenizer=tokenizer,
        client=client,
        permutations=permutations,
        total_permutations=total_permutations,
        batch_size=BATCH_SIZE,
        max_epoch_retries_without_progress=MAX_EPOCH_RETRIES_WITHOUT_PROGRESS,
        batch_cooldown_seconds=BATCH_COOLDOWN_SECONDS,
        patient_model_id=PATIENT_MODEL_ID,
        max_tokens_per_response=MAX_TOKENS_PER_RESPONSE,
        num_utterances=NUM_UTTERANCES,
        temperature_therapist=TEMPERATURE_THERAPIST,
        temperature_patient=TEMPERATURE_PATIENT,
        therapist_system_prompt=therapist_good_system_prompt,
        therapist_init_utterance=therapist_good_init_utterance,
        therapist_max_input_tokens=THERAPIST_MAX_INPUT_TOKENS,
        oom_fallback_max_new_tokens=OOM_FALLBACK_MAX_NEW_TOKENS,
        oom_fallback_temperature=OOM_FALLBACK_TEMPERATURE,
        verbose_responses=verbose_responses,
        verbose_conversation=verbose_conversation,
    )

    print_epoch_completion(
        path_to_save=path_to_save,
        epoch_num=epoch_num,
        total_permutations=total_permutations,
    )


try:
    for epoch in EPOCHS_TO_RUN:
        if epoch == 0 and has_adapters:
            # Disable all adapters to get pure base model behavior
            with therapist_model.disable_adapter():
                _run_single_epoch(epoch, therapist_model, labels_by_epoch,
                                  verbose_responses=False, verbose_conversation=False)
        else:
            _run_single_epoch(epoch, therapist_model, labels_by_epoch,
                              verbose_responses=False, verbose_conversation=False)
finally:
    del therapist_model
    gc.collect()
    torch.cuda.empty_cache()

# ═══════════════════════════════════════════════════════════════════════════════
# Done
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("ALL EPOCHS COMPLETED")
print("=" * 70)
print(f"  Epochs processed:  {EPOCHS_TO_RUN}")
print(f"  Output base dir:   {OUTPUT_BASE_DIR}")
print(f"  Conversations per epoch: {total_permutations}")
print(f"  Batch size:        {BATCH_SIZE}")
print("=" * 70)